# p1804 Implement Trie II (Prefix Tree) 解析笔记

- 题号：p1804
- 题目英文名：Implement Trie II (Prefix Tree)
- 题目中文名：实现 Trie II（前缀树）
- 题目链接：https://leetcode.cn/problems/implement-trie-ii-prefix-tree/
- 题型：前缀树
- 难度：Medium
- 推荐优先级：低
- Java 原代码位置：`solutions-java/leetcode/p1804_implement_trie_ii_prefix_tree/ImplementTrieIIPrefixTree.java`


## 1. 题目一句话总结

这道题要求我们实现一个支持插入、删除、精确计数和前缀计数的 Trie。

本质上考的是如何在前缀树节点里维护 `pass` 和 `end` 两类计数信息。


## 2. 题目理解与约束分析

### 2.1 题目要求

实现一个前缀树，支持：

- `insert(word)`
- `erase(word)`
- `countWordsEqualTo(word)`
- `countWordsStartingWith(prefix)`

### 2.2 输入与输出

- 输入：字符串操作序列
- 输出：对应计数结果
- 返回结果含义：单词出现次数和前缀出现次数

### 2.3 关键约束

- 同一个单词可以插入多次
- 删除时只删一次出现
- 需要支持前缀计数，不只是判断存在性

### 2.4 示例分析

如果先插入：

```text
apple
apple
app
```

那么：

- `countWordsEqualTo("apple") = 2`
- `countWordsStartingWith("app") = 3`


## 3. Java 原代码

```java
package leetcode.p1804_implement_trie_ii_prefix_tree;

import java.util.HashMap;

public class ImplementTrieIIPrefixTree {

    static class Trie {

        static class TrieNode {
            public int pass;
            public int end;
            HashMap<Integer, Trie.TrieNode> nextNodeMap;

            public TrieNode() {
                pass = 0;
                end = 0;
                nextNodeMap = new HashMap<>();
            }
        }

        private final TrieNode root;

        public Trie() {
            root = new Trie.TrieNode();
        }

        public void insert(String word) {
            Trie.TrieNode node = root;
            node.pass++;
            for (int i = 0, path; i < word.length(); i++) {
                path = word.charAt(i);
                if (!node.nextNodeMap.containsKey(path)) {
                    node.nextNodeMap.put(path, new Trie.TrieNode());
                }
                node = node.nextNodeMap.get(path);
                node.pass++;
            }
            node.end++;
        }

        public void erase(String word) {
            if (countWordsEqualTo(word) > 0) {
                Trie.TrieNode node = root;
                Trie.TrieNode next;
                node.pass--;
                for (int i = 0, path; i < word.length(); i++) {
                    path = word.charAt(i);
                    next = node.nextNodeMap.get(path);
                    if (--next.pass == 0) {
                        node.nextNodeMap.remove(path);
                        return;
                    }
                    node = next;
                }
                node.end--;
            }
        }

        public int countWordsEqualTo(String word) {
            Trie.TrieNode node = root;
            for (int i = 0, path; i < word.length(); i++) {
                path = word.charAt(i);
                if (!node.nextNodeMap.containsKey(path)) {
                    return 0;
                }
                node = node.nextNodeMap.get(path);
            }
            return node.end;
        }

        public int countWordsStartingWith(String pre) {
            Trie.TrieNode node = root;
            for (int i = 0, path; i < pre.length(); i++) {
                path = pre.charAt(i);
                if (!node.nextNodeMap.containsKey(path)) {
                    return 0;
                }
                node = node.nextNodeMap.get(path);
            }
            return node.pass;
        }
    }
}
```


## 4. 先从 Java 原方案理解

Java 原方案的关键不只是 Trie 结构本身，而是每个节点维护了两个计数：

- `pass`：有多少单词经过这个节点
- `end`：有多少单词在这个节点结束

同时它没有用固定 26 长数组，而是用了：

```java
HashMap<Integer, TrieNode> nextNodeMap;
```

这样更通用，字符集更大时也适用。

尤其 `erase` 的写法很值得保留：如果某条边对应的下一个节点 `pass` 在减 1 后变成 0，就直接把这条边删掉，整棵子树都能顺便释放逻辑引用。


## 5. 从朴素思路到优化思路

### 5.1 最直接的想法

最直接可以用哈希表存整词出现次数，再遍历所有键去统计某个前缀开头的单词数。

### 5.2 为什么不够好

- 前缀统计会退化成遍历所有单词
- 删除和计数不能充分利用公共前缀

### 5.3 优化方向

Trie 的优势就是把公共前缀合并存储。Java 原方案进一步用 `pass/end` 把“经过多少次”和“结束多少次”都记录下来，于是删除和计数都能高效完成。


## 6. 核心算法讲解

### 6.1 算法名称

- Trie
- pass/end 计数维护

### 6.2 为什么想到这个算法

因为题目不只是判断存在与否，还要统计某个前缀下有多少单词，这正是 Trie 的强项。

### 6.3 关键状态

- `pass`：经过当前节点的单词数量
- `end`：在当前节点结束的单词数量
- `next_node_map`：当前节点到下一个字符节点的映射

### 6.4 步骤拆解

1. `insert` 时沿路径走，沿途 `pass++`，最后 `end++`
2. `countWordsEqualTo` 时走完整个单词路径，看最后节点的 `end`
3. `countWordsStartingWith` 时走完整个前缀路径，看最后节点的 `pass`
4. `erase` 时若单词存在，则沿路径 `pass--`
5. 若某个后继节点 `pass` 变成 0，直接删除这条边并提前结束


In [ ]:
class TrieNode:
    def __init__(self) -> None:
        self.pass_count = 0
        self.end_count = 0
        self.next_node_map: dict[str, TrieNode] = {}


class Trie:
    def __init__(self) -> None:
        self.root = TrieNode()

    def insert(self, word: str) -> None:
        node = self.root
        node.pass_count += 1
        for ch in word:
            if ch not in node.next_node_map:
                node.next_node_map[ch] = TrieNode()
            node = node.next_node_map[ch]
            node.pass_count += 1
        node.end_count += 1

    def erase(self, word: str) -> None:
        if self.countWordsEqualTo(word) > 0:
            node = self.root
            node.pass_count -= 1
            for ch in word:
                next_node = node.next_node_map[ch]
                next_node.pass_count -= 1
                if next_node.pass_count == 0:
                    del node.next_node_map[ch]
                    return
                node = next_node
            node.end_count -= 1

    def countWordsEqualTo(self, word: str) -> int:
        node = self.root
        for ch in word:
            if ch not in node.next_node_map:
                return 0
            node = node.next_node_map[ch]
        return node.end_count

    def countWordsStartingWith(self, prefix: str) -> int:
        node = self.root
        for ch in prefix:
            if ch not in node.next_node_map:
                return 0
            node = node.next_node_map[ch]
        return node.pass_count


## 7. 代码逐段讲解

### 7.1 为什么保留 HashMap 思路

Java 原解特意用 `HashMap<Integer, TrieNode>`，说明它不想把实现限制死在 26 个小写字母。Python 这里也保留这个思路，用字典做后继映射。

### 7.2 `pass` 和 `end`

- `pass_count` 对应 Java 的 `pass`
- `end_count` 对应 Java 的 `end`

前者负责前缀计数，后者负责完整单词计数。

### 7.3 `erase`

这部分最值得看。Java 原解先判断单词是否存在，然后沿路减 `pass`。一旦发现某个后继节点 `pass` 变成 0，就把这条边删掉，后面无需继续。


## 8. 复杂度分析

- 设单词长度为 `L`
- `insert` / `erase` / `countWordsEqualTo` / `countWordsStartingWith` 时间复杂度：`O(L)`
- 空间复杂度：与插入单词总字符数有关


## 9. 易错点

1. 只维护 `end`，不维护 `pass`，导致前缀计数不好做。
2. 删除单词时没有先判断单词是否存在。
3. `erase` 时只减 `end` 不减沿途 `pass`，导致前缀统计错误。
4. 忘了在 `pass` 变成 0 时删除后继边，结构无法正确收缩。


## 10. 面试时怎么讲

可以这样概括：

> 这题我会用 Trie 来做，但和基础版不同，我会在每个节点上维护两个计数：`pass` 和 `end`。
> `pass` 表示有多少单词经过当前节点，`end` 表示有多少单词在当前节点结束。
> 这样完整单词计数看 `end`，前缀计数看 `pass`，删除时则沿路把 `pass` 减掉。
> 如果某个后继节点的 `pass` 归零，就可以直接删掉整条边。


In [ ]:
trie = Trie()
trie.insert('apple')
trie.insert('apple')
trie.insert('app')
print(trie.countWordsEqualTo('apple'))
print(trie.countWordsStartingWith('app'))
trie.erase('apple')
print(trie.countWordsEqualTo('apple'))
print(trie.countWordsStartingWith('app'))


## 11. Demo 输出说明

- 第一次完整计数应输出 `2`。
- 前缀 `app` 计数应输出 `3`。
- 删除一次 `apple` 后，完整计数应降为 `1`，前缀计数应降为 `2`。


## 12. 一句话复盘

> 这题最关键的不是 Trie 结构本身，而是像 Java 原解那样在每个节点同时维护 `pass` 和 `end`。
